# MedGemma reader on a free Kaggle GPU

Runs `python/scripts/11_ai_predict.py` from [dental-omop-nlp](https://github.com/jcbowyer/dental-omop-nlp)
with MedGemma on Kaggle's free GPU, for when Databricks Free Edition has no GPU to give. Same code, same
prompts and the same 240 **synthetic** review notes as the Databricks job; nothing here touches real
patient data.

**Before you start (once):**

1. A Kaggle account with a **verified phone number** (Kaggle requires it for GPUs and internet).
2. Accept the MedGemma licence on Kaggle: open [keras/medgemma](https://www.kaggle.com/models/keras/medgemma)
   and agree to the Health AI Developer Foundations terms. The default backend, `keras` (KerasHub on JAX),
   loads `medgemma_1.5_instruct_4b` from there and needs **no Hugging Face token**. Run from GitHub, the model
   is attached for you; run by hand, attach it here: **Add Input → Models → MedGemma → Keras →
   medgemma_1.5_instruct_4b**.
3. Only for `BACKEND = "hf"` (transformers, as in the first committed results): on Hugging Face, accept the
   terms on [google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it), create a **Read**
   token with a **7 day** expiry, and in this notebook **Add-ons → Secrets → Add a new secret**, label
   `HF_TOKEN`, paste the token there (never in a cell), and tick it for this notebook. A notebook pushed from
   GitHub can't read secrets, so the workflow uses `keras`.
4. In the right-hand panel: **Accelerator → GPU T4 x2** and **Internet → On**. The hf backend runs float32
   across both GPUs (the T4 has no native bfloat16, and Gemma models can overflow in float16); the keras
   backend runs bfloat16 on one (float32 doesn't fit a single T4).

Then run the cells in order. Do the 10-note test first (with `keras` it is compared with the committed Hugging Face results); for the full run use **Save Version → Save & Run All
(Commit)** so it carries on after you close the browser, and download the results from the **Output** tab.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings

In [ ]:
RUNG = "medgemma-1.5-4b-it"   # or medgemma-4b-it (models.yaml)
BACKEND = "keras"             # keras: Kaggle Models, no token; hf: transformers, needs the HF_TOKEN secret
SHOTS = 0                     # few-shot examples per status (0 = zero-shot, as in the README results)
TEST_NOTES = 10               # quick check before the full 240
BRANCH = "main"
FULL_RUN = False              # True: also read all 240 notes (cell 2); set by the GitHub workflow


## Get the code (and, for hf, the Hugging Face token)


In [ ]:
import os

if BACKEND == "hf":
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # read from Kaggle Secrets, never printed
    except Exception as e:
        msg = ("No HF_TOKEN secret on this notebook. On kaggle.com open it, then Add-ons -> Secrets -> "
               "add HF_TOKEN (a Hugging Face Read token) and tick it for this notebook, then run again; "
               f"or set BACKEND = \"keras\", which needs no token. ({type(e).__name__})")
        print(msg, flush=True)   # printed too: Kaggle's IPython can crash formatting the traceback and hide it
        raise SystemExit(msg)
    EXTRA = "llm"
else:
    os.environ["KERAS_BACKEND"] = "jax"   # before anything imports keras
    EXTRA = "keras"

REPO = "/kaggle/temp/dental-omop-nlp"   # outside /kaggle/working, so the clone isn't saved as output
os.makedirs("/kaggle/temp", exist_ok=True)
!git clone --quiet --depth 1 -b {BRANCH} https://github.com/jcbowyer/dental-omop-nlp {REPO}
!pip install --quiet -e "{REPO}/python[{EXTRA}]"
os.environ["DENTAL_TELEMETRY"] = "off"
print("ready:", BACKEND)


## 1. Test on a few notes

Loads the model (a few minutes the first time) and reads `TEST_NOTES` notes. The log says which dtype it
picked (hf: float32 on T4 x2; keras: bfloat16). With `keras`, the test notes are then compared with the
committed Hugging Face results for the same notes: agreement, gate pass rate and accuracy.

In [ ]:
!cd {REPO}/python && python scripts/11_ai_predict.py --rung {RUNG} --backend {BACKEND} --shots {SHOTS} --limit {TEST_NOTES}
if BACKEND != "hf":
    # same notes, committed Hugging Face run: check agreement before trusting a full keras run
    base = f"{REPO}/data/datasets/llm_details_{RUNG}" + (f"+k{SHOTS}" if SHOTS else "") + ".csv"
    if os.path.exists(base):
        !cd {REPO}/python && python scripts/40_compare_backends.py --rung {RUNG} --shots {SHOTS}
    else:
        print("no committed hf run of", RUNG, "with", SHOTS, "shots to compare with")


## 2. Full run: all 240 review notes

Several minutes per 10 notes on T4 x2, so a few hours in all: run it with **Save & Run All (Commit)**. Set `FULL_RUN = True` in the settings first.


In [ ]:
if FULL_RUN:
    !cd {REPO}/python && python scripts/11_ai_predict.py --rung {RUNG} --backend {BACKEND} --shots {SHOTS}
else:
    print("FULL_RUN is False: skipping the 240-note run (set it to True above, or run the workflow with limit 0)")


## 3. Keep the results

Copies the prediction files to `/kaggle/working`, which is what the **Output** tab offers for download.

In [ ]:
import glob, shutil
files = glob.glob(f"{REPO}/data/datasets/*_{RUNG}*.csv")
for f in files:
    if not os.path.basename(f).startswith("simulated_"):
        shutil.copy(f, "/kaggle/working/")
        print("saved", os.path.basename(f))

## Back in the repo

1. Put the downloaded `ai_predictions_<rung>.csv` and `llm_details_<rung>.csv` in `data/datasets/` (a keras
   run's files end in `+keras`, so they sit beside the hf results instead of replacing them).
2. `cd python && python scripts/28_ladder_vs_truth.py && python scripts/35_readme_results.py`
3. Commit: the README and `docs/results.md` then show MedGemma's real (not simulated) scores.